# CBT Cognitive Distortion Classification with DeBERTa + HQDE

**Optimized for Kaggle: 2x T4 GPUs, 4 vCPUs**

This notebook trains an ensemble of DeBERTa models using the HQDE framework for cognitive distortion classification.

## Default Dataset
- **Source**: Hugging Face `danthareja/cognitive-distortion`
- **Task**: classify cognitive distortion labels from therapy-style text
- **Classes**: loaded dynamically from the dataset, including `No Distortion`
- **Fallback**: set `HQDE_DATASET_SOURCE=synthetic` only for offline smoke tests

## Hardware Configuration
- **GPUs**: 2x T4
- **vCPUs**: 4
- **RAM**: ~30GB

## Notes
- Metrics from the Hugging Face dataset are experimental benchmark metrics, not clinical evidence.
- The synthetic fallback is for runtime validation only and should not be reported as a thesis result.


In [ ]:
# Install HQDE package and notebook dependencies
# --no-deps keeps Kaggle's preinstalled CUDA/Torch stack intact.
!pip install -q --no-deps "hqde==0.1.12"
!pip install -q "transformers>=4.30.0" "datasets>=2.14.0" "accelerate>=0.20.0" "sentencepiece>=0.1.99"
!pip install -q "scikit-learn>=1.3.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "seaborn>=0.12.0"

In [ ]:
import os
import json
import random
from contextlib import nullcontext
from importlib.metadata import PackageNotFoundError, version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
except ImportError:  # Older PyTorch fallback
    from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_cosine_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

try:
    print(f"HQDE package version: {version('hqde')}")
except PackageNotFoundError:
    print("HQDE package is not installed. Run the install cell first.")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No CUDA GPU found. The notebook will run in CPU-compatible mode for smoke tests.")

In [ ]:
# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Configuration
class Config:
    # Runtime knobs
    quick_test = os.environ.get("HQDE_QUICK_TEST", "0") == "1"
    detected_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    num_gpus = min(detected_gpus, 2)
    device = "cuda" if num_gpus > 0 else "cpu"

    # Model
    model_name = os.environ.get("HQDE_MODEL_NAME", "microsoft/deberta-v3-base")
    num_classes = 10  # overwritten when using the Hugging Face dataset
    max_length = 64 if quick_test else 256

    # Data source
    dataset_source = os.environ.get("HQDE_DATASET_SOURCE", "hf").strip().lower()
    hf_dataset_name = os.environ.get("HQDE_HF_DATASET", "danthareja/cognitive-distortion")
    max_train_samples = int(os.environ.get("HQDE_MAX_TRAIN_SAMPLES", "128" if quick_test else "0"))
    max_eval_samples = int(os.environ.get("HQDE_MAX_EVAL_SAMPLES", "64" if quick_test else "0"))

    # Training
    num_workers = 1 if quick_test or num_gpus == 0 else min(4, num_gpus * 2)
    batch_size = int(os.environ.get("HQDE_BATCH_SIZE", "2" if quick_test or num_gpus == 0 else "8"))
    num_epochs = int(os.environ.get("HQDE_NUM_EPOCHS", "1" if quick_test else "15"))
    learning_rate = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    max_grad_norm = 1.0

    # Ensemble diversity
    dropout_rates = [0.1, 0.15, 0.2, 0.25]
    learning_rates = [1.5e-5, 2e-5, 2.5e-5, 3e-5]

    # DataLoader/runtime
    # Kaggle/Python 3.12 notebooks can crash DataLoader subprocesses during
    # repeated train/eval loops, so use single-process loading by default.
    dataloader_workers = int(os.environ.get("HQDE_DATALOADER_WORKERS", "0"))
    pin_memory = os.environ.get("HQDE_PIN_MEMORY", "0") == "1" and num_gpus > 0
    use_amp = num_gpus > 0

    # Data
    test_size = 0.2
    val_size = 0.1

    # Output
    output_dir = "./cbt_output"

    # Class names
    distortion_names = [
        "All-or-Nothing Thinking",
        "Overgeneralization",
        "Mental Filter",
        "Disqualifying the Positive",
        "Jumping to Conclusions",
        "Magnification/Catastrophizing",
        "Emotional Reasoning",
        "Should Statements",
        "Labeling",
        "Personalization",
    ]

config = Config()
os.makedirs(config.output_dir, exist_ok=True)

print("Configuration set")
print(f"  Model: {config.model_name}")
print(f"  Runtime: device={config.device}, detected_gpus={config.detected_gpus}, active_gpus={config.num_gpus}")
print(f"  Dataset source: {config.dataset_source}")
print(f"  HF dataset: {config.hf_dataset_name if config.dataset_source in {'hf', 'huggingface'} else 'not used'}")
print(f"  Ensemble workers: {config.num_workers}")
print(f"  Batch size per worker: {config.batch_size}")
print(f"  Epochs: {config.num_epochs}")
print(f"  AMP enabled: {config.use_amp}")
print(f"  DataLoader workers: {config.dataloader_workers}")
print(f"  Pin memory: {config.pin_memory}")
if config.quick_test:
    print("  Quick test mode enabled via HQDE_QUICK_TEST=1")

In [ ]:
# Dataset loading
SYNTHETIC_DISTORTION_NAMES = [
    "All-or-Nothing Thinking",
    "Overgeneralization",
    "Mental Filter",
    "Disqualifying the Positive",
    "Jumping to Conclusions",
    "Magnification/Catastrophizing",
    "Emotional Reasoning",
    "Should Statements",
    "Labeling",
    "Personalization",
]


def clean_text(value):
    if value is None:
        return ""
    return str(value).strip()


def load_hf_cognitive_distortion_dataset(dataset_name):
    raw = load_dataset(dataset_name)
    if "train" not in raw:
        raise ValueError(f"{dataset_name} must provide a train split")

    label_feature = raw["train"].features.get("dominant_distortion")
    if label_feature is None or not hasattr(label_feature, "names"):
        raise ValueError("Expected a ClassLabel column named 'dominant_distortion'")

    label_names = list(label_feature.names)

    def split_to_frame(split_name):
        if split_name not in raw:
            return pd.DataFrame(columns=["id", "text", "label", "distortion_name"])

        records = []
        for row in raw[split_name]:
            label = int(row["dominant_distortion"])
            text = (
                clean_text(row.get("distorted_part"))
                or clean_text(row.get(" patient_question"))
                or clean_text(row.get("patient_question"))
            )
            if not text:
                continue

            records.append({
                "id": row.get("id"),
                "text": text,
                "label": label,
                "distortion_name": label_names[label],
            })

        frame = pd.DataFrame(records)
        if frame.empty:
            raise ValueError(f"No usable rows found in split: {split_name}")
        return frame.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    train_source_df = split_to_frame("train")
    test_df = split_to_frame("test")

    # Prevent exact-text leakage from the published train split into the test split.
    train_texts = set(train_source_df["text"])
    test_df = test_df[~test_df["text"].isin(train_texts)].reset_index(drop=True)

    return train_source_df, test_df, label_names


def generate_synthetic_cbt_dataset():
    """Offline smoke-test fallback. Do not report these metrics as benchmark results."""
    templates = {
        0: "I got a B+ on my exam, so I am a complete failure unless everything is perfect.",
        1: "I made one mistake at work, so I always mess everything up and never do anything right.",
        2: "My review was mostly positive, but I can only think about one critical comment.",
        3: "I got promoted, but it does not count because they probably just needed someone.",
        4: "My friend did not text back, so she must be angry and ending the friendship.",
        5: "I made a small email mistake, so my whole career is probably ruined.",
        6: "I feel incompetent, so that must mean I really am incompetent.",
        7: "I should handle everything perfectly and must never need help.",
        8: "I forgot a name, so I am a terrible person and a loser.",
        9: "My team missed the deadline, and it is entirely my fault even though many people were involved.",
    }

    rows = []
    for label, text in templates.items():
        for i in range(10):
            rows.append({
                "id": f"synthetic-{label}-{i}",
                "text": f"{text} Case variant {i + 1}.",
                "label": label,
                "distortion_name": SYNTHETIC_DISTORTION_NAMES[label],
            })
    return pd.DataFrame(rows)


def stratified_cap(frame, max_rows, seed=42):
    if not max_rows or len(frame) <= max_rows:
        return frame.reset_index(drop=True)

    labels = sorted(frame["label"].unique())
    per_class = max(max_rows // len(labels), 1)
    selected = []
    used_indices = set()

    for label in labels:
        group = frame[frame["label"] == label]
        take = min(len(group), per_class)
        sample = group.sample(n=take, random_state=seed + int(label))
        selected.append(sample)
        used_indices.update(sample.index.tolist())

    capped = pd.concat(selected, axis=0)
    remaining = max_rows - len(capped)
    if remaining > 0:
        rest = frame.drop(index=list(used_indices), errors="ignore")
        if not rest.empty:
            capped = pd.concat([
                capped,
                rest.sample(n=min(remaining, len(rest)), random_state=seed),
            ], axis=0)

    return capped.sample(frac=1.0, random_state=seed).reset_index(drop=True)


In [ ]:
# Load and split data
if config.dataset_source in {"hf", "huggingface"}:
    train_source_df, test_df, label_names = load_hf_cognitive_distortion_dataset(config.hf_dataset_name)
    config.distortion_names = label_names
    config.num_classes = len(label_names)

    train_source_df = stratified_cap(train_source_df, config.max_train_samples, seed=42)
    test_df = stratified_cap(test_df, config.max_eval_samples, seed=44)

    if train_source_df["label"].value_counts().min() < 2:
        raise ValueError("Not enough samples per class for a stratified train/validation split")

    train_df, val_df = train_test_split(
        train_source_df,
        test_size=config.val_size,
        random_state=42,
        stratify=train_source_df["label"],
    )
    val_df = stratified_cap(val_df, config.max_eval_samples, seed=43)
    df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print(f"Dataset loaded from Hugging Face: {config.hf_dataset_name}")
else:
    config.distortion_names = SYNTHETIC_DISTORTION_NAMES
    config.num_classes = len(config.distortion_names)
    df = generate_synthetic_cbt_dataset()
    train_df, temp_df = train_test_split(
        df,
        test_size=config.test_size + config.val_size,
        random_state=42,
        stratify=df["label"],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=config.test_size / (config.test_size + config.val_size),
        random_state=42,
        stratify=temp_df["label"],
    )
    print("Synthetic smoke-test dataset loaded. Do not report these metrics as benchmark results.")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Dataset ready: {len(df)} rows across {config.num_classes} classes")
print(f"  Training: {len(train_df)} samples")
print(f"  Validation: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")

print("\nClass distribution:")
for label in sorted(df["label"].unique()):
    count = int((df["label"] == label).sum())
    name = config.distortion_names[label]
    print(f"  {label}: {name:30s} - {count} samples")

print("\nExact text overlap checks:")
print(f"  Train/validation: {len(set(train_df['text']).intersection(set(val_df['text'])))}")
print(f"  Train/test: {len(set(train_df['text']).intersection(set(test_df['text'])))}")
print(f"  Validation/test: {len(set(val_df['text']).intersection(set(test_df['text'])))}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
print(f"\nTokenizer loaded: {config.model_name}")

# Dataset class
class CBTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

# Create datasets
train_dataset = CBTDataset(train_df["text"].values, train_df["label"].values, tokenizer, config.max_length)
val_dataset = CBTDataset(val_df["text"].values, val_df["label"].values, tokenizer, config.max_length)
test_dataset = CBTDataset(test_df["text"].values, test_df["label"].values, tokenizer, config.max_length)

# Create data loaders
loader_kwargs = {
    "num_workers": config.dataloader_workers,
    "pin_memory": config.pin_memory,
}

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size * 2, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 2, shuffle=False, **loader_kwargs)

print("Datasets and loaders created")


In [ ]:
# DeBERTa Classifier
class DeBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout_rate=0.1):
        super(DeBERTaClassifier, self).__init__()
        
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.hidden_dropout_prob = dropout_rate
        self.config.attention_probs_dropout_prob = dropout_rate
        self.config.dtype = torch.float32
        
        # Keep trainable parameters in FP32. AMP autocast will use FP16 for
        # selected CUDA operations, while GradScaler expects FP32 gradients.
        try:
            self.deberta = AutoModel.from_pretrained(
                model_name,
                config=self.config,
                dtype=torch.float32,
            )
        except TypeError:
            self.deberta = AutoModel.from_pretrained(
                model_name,
                config=self.config,
                torch_dtype=torch.float32,
            )
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.config.hidden_size, num_classes)
        
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        self.float()
    
    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

print("✓ Model architecture defined")

In [ ]:
# Ensemble Worker
def make_grad_scaler(enabled: bool):
    try:
        return GradScaler("cuda", enabled=enabled)
    except TypeError:
        return GradScaler(enabled=enabled)


def amp_context(device: torch.device, enabled: bool):
    if not enabled:
        return nullcontext()
    try:
        return autocast(device_type=device.type, dtype=torch.float16, enabled=enabled)
    except TypeError:
        return autocast(enabled=enabled)


class EnsembleWorker:
    def __init__(self, worker_id, config, device):
        self.worker_id = worker_id
        self.config = config
        self.device = torch.device(device)

        self.dropout_rate = config.dropout_rates[worker_id % len(config.dropout_rates)]
        self.learning_rate = config.learning_rates[worker_id % len(config.learning_rates)]
        self.use_amp = bool(config.use_amp and self.device.type == "cuda")

        self.model = DeBERTaClassifier(config.model_name, config.num_classes, self.dropout_rate).to(self.device)
        if self.use_amp:
            self.model.float()
        first_param = next(self.model.parameters())
        print(
            f"Worker {worker_id}: dropout={self.dropout_rate}, lr={self.learning_rate}, "
            f"device={self.device}, amp={self.use_amp}, param_dtype={first_param.dtype}"
        )

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.learning_rate,
            weight_decay=config.weight_decay,
        )
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = make_grad_scaler(self.use_amp)

        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []

    def train_epoch(self, train_loader, scheduler):
        self.model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Worker {self.worker_id} Training")
        for batch in pbar:
            input_ids = batch['input_ids'].to(self.device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
            labels = batch['labels'].to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            with amp_context(self.device, self.use_amp):
                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)

            if self.use_amp:
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
                self.optimizer.step()

            scheduler.step()

            batch_size = labels.size(0)
            total_loss += loss.detach().item() * batch_size
            _, predicted = torch.max(logits.detach(), 1)
            total += batch_size
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({'loss': f'{loss.detach().item():.4f}', 'acc': f'{100 * correct / max(total, 1):.2f}%'})

        return total_loss / max(total, 1), 100 * correct / max(total, 1)

    def evaluate(self, val_loader):
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Worker {self.worker_id} Validation"):
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                labels = batch['labels'].to(self.device, non_blocking=True)

                logits = self.model(input_ids, attention_mask)
                loss = self.criterion(logits, labels)

                batch_size = labels.size(0)
                total_loss += loss.detach().item() * batch_size
                _, predicted = torch.max(logits, 1)
                total += batch_size
                correct += (predicted == labels).sum().item()

        return total_loss / max(total, 1), 100 * correct / max(total, 1)

    def predict(self, data_loader):
        self.model.eval()
        all_logits = []

        with torch.no_grad():
            for batch in data_loader:
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                logits = self.model(input_ids, attention_mask)
                all_logits.append(logits.cpu())

        return torch.cat(all_logits, dim=0)

print("Ensemble worker class defined")

In [ ]:
# Create ensemble workers
def resolve_worker_devices(config):
    if config.num_gpus > 0:
        return [torch.device(f"cuda:{i % config.num_gpus}") for i in range(config.num_workers)]
    return [torch.device("cpu") for _ in range(config.num_workers)]


workers = []
devices = resolve_worker_devices(config)

print(f"Creating {config.num_workers} ensemble workers:\n")
for i, device in enumerate(devices):
    worker = EnsembleWorker(i, config, device)
    workers.append(worker)

print(f"\n{config.num_workers} workers created")

In [ ]:
# Training loop
print(f"Starting training for {config.num_epochs} epochs\n")

best_ensemble_acc = 0.0
num_training_steps = max(len(train_loader) * config.num_epochs, 1)
num_warmup_steps = int(num_training_steps * config.warmup_ratio)

schedulers = {
    worker.worker_id: get_cosine_schedule_with_warmup(
        worker.optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    for worker in workers
}

for epoch in range(config.num_epochs):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{config.num_epochs}")
    print(f"{'='*80}")

    # Train each worker
    for worker in workers:
        scheduler = schedulers[worker.worker_id]
        train_loss, train_acc = worker.train_epoch(train_loader, scheduler)
        val_loss, val_acc = worker.evaluate(val_loader)

        worker.train_losses.append(train_loss)
        worker.val_losses.append(val_loss)
        worker.val_accuracies.append(val_acc)

        print(f"\nWorker {worker.worker_id}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

    # Ensemble evaluation
    print(f"\nEnsemble Evaluation:")
    all_worker_logits = []

    for worker in workers:
        logits = worker.predict(val_loader)
        all_worker_logits.append(logits)

    ensemble_logits = torch.stack(all_worker_logits).mean(dim=0)
    ensemble_preds = torch.argmax(ensemble_logits, dim=1).numpy()
    true_labels = val_df['label'].values

    ensemble_acc = accuracy_score(true_labels, ensemble_preds) * 100
    ensemble_f1 = f1_score(true_labels, ensemble_preds, average='weighted') * 100

    print(f"  Ensemble Accuracy: {ensemble_acc:.2f}%")
    print(f"  Ensemble F1-Score: {ensemble_f1:.2f}%")

    if ensemble_acc > best_ensemble_acc:
        best_ensemble_acc = ensemble_acc
        print("  New best!")

print(f"\nTraining completed! Best validation accuracy: {best_ensemble_acc:.2f}%")

In [ ]:
# Final evaluation on test set
print("\nFinal Evaluation on Test Set\n")

test_worker_logits = []
for worker in workers:
    logits = worker.predict(test_loader)
    test_worker_logits.append(logits)

ensemble_test_logits = torch.stack(test_worker_logits).mean(dim=0)
ensemble_test_preds = torch.argmax(ensemble_test_logits, dim=1).numpy()
test_true_labels = test_df['label'].values

test_acc = accuracy_score(test_true_labels, ensemble_test_preds) * 100
test_f1 = f1_score(test_true_labels, ensemble_test_preds, average='weighted') * 100

print(f"{'='*80}")
print(f"FINAL TEST RESULTS")
print(f"{'='*80}")
print(f"Ensemble Test Accuracy: {test_acc:.2f}%")
print(f"Ensemble Test F1-Score: {test_f1:.2f}%")

print(f"\nClassification Report:")
print(classification_report(test_true_labels, ensemble_test_preds, labels=list(range(config.num_classes)), target_names=config.distortion_names, digits=3, zero_division=0))

# Confusion matrix
cm = confusion_matrix(test_true_labels, ensemble_test_preds, labels=list(range(config.num_classes)))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f"{i}" for i in range(config.num_classes)],
            yticklabels=[f"{i}" for i in range(config.num_classes)])
plt.title('Confusion Matrix - DeBERTa HQDE Ensemble')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{config.output_dir}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Confusion matrix saved")

# Per-class accuracy
print(f"\nPer-Class Accuracy:")
for i in range(config.num_classes):
    class_mask = test_true_labels == i
    if class_mask.sum() > 0:
        class_acc = (ensemble_test_preds[class_mask] == test_true_labels[class_mask]).mean() * 100
        print(f"  {i}: {config.distortion_names[i]:35s} - {class_acc:.2f}%")

# Individual worker performance
print(f"\nIndividual Worker Test Accuracy:")
for i, worker in enumerate(workers):
    worker_logits = test_worker_logits[i]
    worker_preds = torch.argmax(worker_logits, dim=1).numpy()
    worker_acc = accuracy_score(test_true_labels, worker_preds) * 100
    print(f"  Worker {i}: {worker_acc:.2f}%")

print(f"\n{'='*80}")
print(f"✅ COMPLETE!")
print(f"{'='*80}")
print(f"Best Validation Accuracy: {best_ensemble_acc:.2f}%")
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Final Test F1-Score: {test_f1:.2f}%")